# reg_with_dummies

> 对应代码：`EconometricsCode/code_in_notes/Chap.1/reg_with_dummies.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.1`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.1")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


读入 CHFS 个人层面数据，把月收入 a3109 乘以 12 得到年收入 p_income。

In [2]:
%%stata
use ../datasets/chfs2017_ind.dta, clear
gen p_income = a3109*12


. use ../datasets/chfs2017_ind.dta, clear
(chfs2017 ind. data, add demograhic info)

. gen p_income = a3109*12
(93,529 missing values generated)

. 


tab 配合 gen(edu) 选项，按教育程度 a2012 的每个取值自动生成一组虚拟变量 edu1、edu2、…：edu_j=1 当且仅当个体属于第 j 类教育水平。

In [3]:
%%stata
tab a2012, gen(edu)


     文化程 |
         度 |      Freq.     Percent        Cum.
------------+-----------------------------------
          1 |     11,414       10.47       10.47
          2 |     23,729       21.76       32.22
          3 |     32,805       30.08       62.30
          4 |     15,452       14.17       76.47
          5 |      6,164        5.65       82.12
          6 |      8,533        7.82       89.94
          7 |      9,845        9.03       98.97
          8 |        977        0.90       99.87
          9 |        145        0.13      100.00
------------+-----------------------------------
      Total |    109,064      100.00


第一种设定：放入全部虚拟变量并保留截距。由于所有虚拟变量之和恒等于 1，与截距项完全共线，Stata 会自动丢掉一个类别作为基准组；于是每个保留虚拟变量的系数都解释为相应教育组相对于基准组的平均收入差。

In [4]:
%%stata
reg p_income edu*
outreg2 using reg_with_dummies.tex, replace


. reg p_income edu*
note: edu9 omitted because of collinearity.

      Source |       SS           df       MS      Number of obs   =    33,434
-------------+----------------------------------   F(8, 33425)     =    740.24
       Model |  8.8793e+12         8  1.1099e+12   Prob > F        =    0.0000
    Residual |  5.0117e+13    33,425  1.4994e+09   R-squared       =    0.1505
-------------+----------------------------------   Adj R-squared   =    0.1503
       Total |  5.8997e+13    33,433  1.7646e+09   Root MSE        =     38722

------------------------------------------------------------------------------
    p_income | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        edu1 |  -100600.7   4079.854   -24.66   0.000    -108597.3   -92604.02
        edu2 |   -96301.7   3845.039   -25.05   0.000    -103838.1   -88765.29
        edu3 |  -91941.83   3815.994   -24.09   0.000    -99421.

第二种设定：用 noconstant 去掉截距、同时放入全部虚拟变量。此时不存在共线性问题，每个虚拟变量的系数直接等于该教育组的平均收入——虚拟变量回归本质上就是分组均值。

In [5]:
%%stata
reg p_income edu*, noconstant
outreg2 using reg_with_dummies.tex, append


. reg p_income edu*, noconstant

      Source |       SS           df       MS      Number of obs   =    33,434
-------------+----------------------------------   F(9, 33425)     =   5524.40
       Model |  7.4549e+13         9  8.2833e+12   Prob > F        =    0.0000
    Residual |  5.0117e+13    33,425  1.4994e+09   R-squared       =    0.5980
-------------+----------------------------------   Adj R-squared   =    0.5979
       Total |  1.2467e+14    33,434  3.7287e+09   Root MSE        =     38722

------------------------------------------------------------------------------
    p_income | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        edu1 |   26115.86   1492.625    17.50   0.000     23190.26    29041.46
        edu2 |   30414.84   605.8461    50.20   0.000     29227.36    31602.32
        edu3 |   34774.71   380.1764    91.47   0.000     34029.55    35519.87
        edu4 |   

第三种设定：保留截距、但手动剔除 edu1（第一组作为基准组）。这是与第一种等价、但基准组由我们显式指定的写法，系数同样解释为相对于基准组的差异。

In [6]:
%%stata
reg p_income edu2-edu9
outreg2 using reg_with_dummies.tex, append


. reg p_income edu2-edu9

      Source |       SS           df       MS      Number of obs   =    33,434
-------------+----------------------------------   F(8, 33425)     =    740.24
       Model |  8.8793e+12         8  1.1099e+12   Prob > F        =    0.0000
    Residual |  5.0117e+13    33,425  1.4994e+09   R-squared       =    0.1505
-------------+----------------------------------   Adj R-squared   =    0.1503
       Total |  5.8997e+13    33,433  1.7646e+09   Root MSE        =     38722

------------------------------------------------------------------------------
    p_income | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        edu2 |   4298.978   1610.894     2.67   0.008     1141.569    7456.386
        edu3 |   8658.852   1540.281     5.62   0.000     5639.848    11677.86
        edu4 |    12664.7   1595.959     7.94   0.000      9536.56    15792.83
        edu5 |   14674.7